# Full test-set evaluation: ResUNet and data-consistency post-processing


## Prepare


In [1]:
#============= Globals ==========
import torch
from mri_dl import (
    ModelConfig,
    MRIUndersampledDataset,
    ResidualUNet,
    evaluate_and_save_results,
    load_checkpoint,
)
from torch.utils.data import DataLoader
import os
from pathlib import Path
import pandas as pd
HPC = False
#============= main ==========
#set here the path to the course data brain age folder
data_path = '../../../../../../../mri_dataset/brain_age'

data_path = os.path.abspath(data_path)
print(f'Data path: {data_path}')
os.chdir(data_path)

experiment_config = ModelConfig(
    plane="coronal",
    retain_ratio=0.30,
    #epochs=60,
    epochs=2,
    batch_size=32,
    learning_rate=5e-4,
    weight_decay=1e-4,
    random_seed=42,
    num_workers=0 if os.name == "nt" else 4,
    data_consistency_enabled=True,
    per_image_csv_logging=True,
    #default is to use the split data set created in the previous notebook
    data_root= Path(os.getcwd() + r'../../../../../../../notebook_5_undersampled_dataset_split').resolve(),
    #set here the path to the required results folder
    result_root=Path(os.getcwd() + r'../../../../../../../notebook_5_undersampled_results').resolve(),)


Data path: C:\mri_dataset\brain_age


## Load test dataset and trained model


In [2]:
test_dataset = MRIUndersampledDataset(
    experiment_config.data_root / "test",
    csv_name="samples.csv",
    plane=experiment_config.plane.capitalize(),
    retain_ratio=experiment_config.retain_ratio,
    load_mask=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)
loaded_model, checkpoint = load_checkpoint(
    checkpoint_path=experiment_config.result_dir / "best_model.pt",
    model_class=ResidualUNet,
)
print(f"Test samples: {len(test_dataset)}")
print("Best epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["validation_loss"])


Test samples: 240
Best epoch: 65
Best validation loss: 0.028095229342579842


## Evaluate the full test set
This step saves per-image results to `test_per_image.csv`.


In [3]:
if len(test_dataset) == 0:
    raise ValueError("test_dataset is empty")
per_sample_df = evaluate_and_save_results(
    model=loaded_model,
    test_loader=test_loader,
    output_root=Path("results"),
    plane="Coronal",
    retain_ratio=0.30,
    checkpoint_name="coronal_r30_delta_best.pt",
    epoch=checkpoint.get("epoch"),
)
display(per_sample_df.head())
print(per_sample_df.shape)


Saved rows: 240
Output path: results\coronal\retain_30\test_per_image.csv
Unique volumes: 60
Plane: Coronal
Retain ratio: 0.3


,sample_id,volume_id,plane,slice_index,retain_ratio,mask_id,masked_rows,psnr_undersampled,psnr_resunet,psnr_resunet_dc,...,resunet_improved_ssim,dc_improved_ssim,final_improved_ssim_vs_us,checkpoint_name,epoch,split,normalization_method,data_range,metric_policy,inference_time_ms
0,3016_row001272_coronal_s0058_r30_u000,3016,coronal,58,0.3,mask_3016_row001272_coronal_s0058_r30_u000,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",23.839050,26.048755,27.324954,...,True,False,True,coronal_r30_delta_best.pt,65,test,clamp_to_unit_range,"reference_min_max (PSNR/SSIM), no per-image re...","PSNR_ROI(reference>0), SSIM_full_frame",0.3421
1,3016_row001272_coronal_s0058_r30_u001,3016,coronal,58,0.3,mask_3016_row001272_coronal_s0058_r30_u001,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 10, 12, 13, 14, 15...",23.860976,26.285236,28.060402,...,True,False,True,coronal_r30_delta_best.pt,65,test,clamp_to_unit_range,"reference_min_max (PSNR/SSIM), no per-image re...","PSNR_ROI(reference>0), SSIM_full_frame",0.3384
2,3016_row001272_coronal_s0076_r30_u000,3016,coronal,76,0.3,mask_3016_row001272_coronal_s0076_r30_u000,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",22.297618,24.986925,26.425717,...,True,False,True,coronal_r30_delta_best.pt,65,test,clamp_to_unit_range,"reference_min_max (PSNR/SSIM), no per-image re...","PSNR_ROI(reference>0), SSIM_full_frame",0.3304
3,3016_row001272_coronal_s0076_r30_u001,3016,coronal,76,0.3,mask_3016_row001272_coronal_s0076_r30_u001,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",22.700927,24.901257,26.132126,...,True,False,True,coronal_r30_delta_best.pt,65,test,clamp_to_unit_range,"reference_min_max (PSNR/SSIM), no per-image re...","PSNR_ROI(reference>0), SSIM_full_frame",0.3064
4,3160_row000068_coronal_s0067_r30_u000,3160,coronal,67,0.3,mask_3160_row000068_coronal_s0067_r30_u000,"[1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 1...",22.297047,23.713869,24.578452,...,True,False,True,coronal_r30_delta_best.pt,65,test,clamp_to_unit_range,"reference_min_max (PSNR/SSIM), no per-image re...","PSNR_ROI(reference>0), SSIM_full_frame",0.3002


(240, 32)


## Summary table (mean/std)


In [4]:
summary_rows = [
    {
        "Method": "undersampled image",
        "Mean PSNR": per_sample_df["psnr_undersampled"].mean(),
        "STD PSNR": per_sample_df["psnr_undersampled"].std(ddof=1),
        "Mean SSIM": per_sample_df["ssim_undersampled"].mean(),
        "STD SSIM": per_sample_df["ssim_undersampled"].std(ddof=1),
    },
    {
        "Method": "ResUnet results",
        "Mean PSNR": per_sample_df["psnr_resunet"].mean(),
        "STD PSNR": per_sample_df["psnr_resunet"].std(ddof=1),
        "Mean SSIM": per_sample_df["ssim_resunet"].mean(),
        "STD SSIM": per_sample_df["ssim_resunet"].std(ddof=1),
    },
    {
        "Method": "ResUnet+data consistency (post processing) results",
        "Mean PSNR": per_sample_df["psnr_resunet_dc"].mean(),
        "STD PSNR": per_sample_df["psnr_resunet_dc"].std(ddof=1),
        "Mean SSIM": per_sample_df["ssim_resunet_dc"].mean(),
        "STD SSIM": per_sample_df["ssim_resunet_dc"].std(ddof=1),
    },
]
summary_df = pd.DataFrame(summary_rows).set_index("Method")
display(summary_df.style.format(precision=4))


,Mean PSNR,STD PSNR,Mean SSIM,STD SSIM
Method,,,,
undersampled image,23.5952,1.5250,0.6601,0.0508
ResUnet results,25.6765,1.6197,0.8782,0.0348
ResUnet+data consistency (post processing) results,27.1467,1.6711,0.8120,0.0372


## Average deltas per step


In [5]:
delta_df = pd.DataFrame(
    [
        {
            "Step": "delta_CNN (ResUnet - undersampled)",
            "Mean Delta PSNR": per_sample_df["psnr_gain_resunet_vs_us"].mean(),
            "Mean Delta SSIM": per_sample_df["ssim_gain_resunet_vs_us"].mean(),
        },
        {
            "Step": "delta_consistency (ResUnet+DC - ResUnet)",
            "Mean Delta PSNR": per_sample_df["psnr_gain_dc_vs_resunet"].mean(),
            "Mean Delta SSIM": per_sample_df["ssim_gain_dc_vs_resunet"].mean(),
        },
    ]
).set_index("Step")
display(delta_df.style.format(precision=4))


,Mean Delta PSNR,Mean Delta SSIM
Step,,
delta_CNN (ResUnet - undersampled),2.0812,0.2181
delta_consistency (ResUnet+DC - ResUnet),1.4702,-0.0662


## Post-processing outcome counts


In [6]:
total = len(per_sample_df)
psnr_improved = per_sample_df["dc_improved_psnr"]
ssim_improved = per_sample_df["dc_improved_ssim"]
psnr_up_ssim_down = per_sample_df["dc_improved_psnr"] & (~per_sample_df["dc_improved_ssim"])
outcome_df = pd.DataFrame(
    [
        {
            "Condition": "Post processing improved PSNR",
            "Count": int(psnr_improved.sum()),
            "Percentage (%)": 100.0 * float(psnr_improved.mean()),
        },
        {
            "Condition": "Post processing improved SSIM",
            "Count": int(ssim_improved.sum()),
            "Percentage (%)": 100.0 * float(ssim_improved.mean()),
        },
        {
            "Condition": "PSNR improved while SSIM reduced",
            "Count": int(psnr_up_ssim_down.sum()),
            "Percentage (%)": 100.0 * float(psnr_up_ssim_down.mean()),
        },
    ]
).set_index("Condition")
print(f"Total evaluated images: {total}")
display(outcome_df.style.format({"Count": "{:.0f}", "Percentage (%)": "{:.2f}"}))


Total evaluated images: 240


,Count,Percentage (%)
Condition,,
Post processing improved PSNR,240,100.00
Post processing improved SSIM,20,8.33
PSNR improved while SSIM reduced,220,91.67
